In [1]:
import httpx
import pandas as pd


URL = (
    "https://opendata.paris.fr/api/explore/v2.1/catalog/"
    "datasets/referentiel-comptages-routiers/records"
)

response = httpx.get(
    URL,
    params={"limit": 20},
    timeout=30,
)

response.raise_for_status()

payload = response.json()

df = pd.DataFrame(payload["results"])

print("Shape:", df.shape)

print("\n=== Columns ===")
print(df.columns.tolist())

print("\n=== Dtypes ===")
print(df.dtypes)

print("\n=== Sample ===")
print(df.head().to_string())

print("\n=== Missing values ===")
print(df.isna().sum())

Shape: (20, 10)

=== Columns ===
['iu_ac', 'date_debut', 'date_fin', 'libelle', 'iu_nd_aval', 'libelle_nd_aval', 'iu_nd_amont', 'libelle_nd_amont', 'geo_point_2d', 'geo_shape']

=== Dtypes ===
iu_ac                  str
date_debut             str
date_fin               str
libelle                str
iu_nd_aval           int64
libelle_nd_aval        str
iu_nd_amont          int64
libelle_nd_amont       str
geo_point_2d        object
geo_shape           object
dtype: object

=== Sample ===
  iu_ac                 date_debut                   date_fin                   libelle  iu_nd_aval        libelle_nd_aval  iu_nd_amont              libelle_nd_amont                                            geo_point_2d                                                                                                                                                                                                                                                      geo_shape
0  6839  2014-04-16T02:00:00+0

In [2]:
import httpx
import pandas as pd


URL = (
    "https://opendata.paris.fr/api/explore/v2.1/catalog/"
    "datasets/referentiel-comptages-routiers/records"
)

ROAD_IDS = [
    "4632", "4634", "1029", "1030", "1031",
    "1032", "1067", "1072", "4630", "4633",
    "4637", "985", "1033", "1034", "1111",
    "1112", "1222", "1043", "1044", "1038",
]


rows = []

for road_id in ROAD_IDS:
    response = httpx.get(
        URL,
        params={
            "where": f'iu_ac="{road_id}"',
            "limit": 100,
        },
        timeout=30,
    )

    response.raise_for_status()

    results = response.json()["results"]

    for row in results:
        rows.append(row)


df_ref = pd.DataFrame(rows)

print("Rows:", len(df_ref))

if not df_ref.empty:
    print("Unique roads:", df_ref["iu_ac"].nunique())

    print("\n=== Versions per road ===")
    print(
        df_ref.groupby("iu_ac")
        .size()
        .sort_values(ascending=False)
    )

    print("\n=== Reference ===")
    print(
        df_ref[
            [
                "iu_ac",
                "libelle",
                "date_debut",
                "date_fin",
                "iu_nd_amont",
                "iu_nd_aval",
            ]
        ]
        .sort_values(["iu_ac", "date_debut"])
        .to_string(index=False)
    )

    found = set(df_ref["iu_ac"].astype(str))
    missing = sorted(set(ROAD_IDS) - found)

    print("\nMissing road IDs:")
    print(missing)

Rows: 22
Unique roads: 20

=== Versions per road ===
iu_ac
1067    2
1030    2
1029    1
1111    1
4637    1
4634    1
4633    1
4632    1
4630    1
1222    1
1112    1
1072    1
1044    1
1043    1
1038    1
1034    1
1033    1
1032    1
1031    1
985     1
dtype: int64

=== Reference ===
iu_ac              libelle                date_debut                  date_fin  iu_nd_amont  iu_nd_aval
 1029 Bd_Gal_Martial_Valin 1996-10-03T02:00:00+00:00 2023-01-01T01:00:00+00:00          573         575
 1030 Bd_Gal_Martial_Valin 1996-10-03T02:00:00+00:00 2023-01-01T01:00:00+00:00          575         573
 1030 Bd_Gal_Martial_Valin 2005-01-01T01:00:00+00:00 2019-06-01T02:00:00+00:00          575         573
 1031 Bd_Gal_Martial_Valin 1996-10-03T02:00:00+00:00 2023-01-01T01:00:00+00:00          575         576
 1032 Bd_Gal_Martial_Valin 1996-10-03T02:00:00+00:00 2023-01-01T01:00:00+00:00          576         575
 1033 Bd_Gal_Martial_Valin 1996-10-03T02:00:00+00:00 2023-01-01T01:00:00+00:00       